# 数据处理

## 读取数据
- 建立数据清单
- 训练集和验证集都是`list`，元素是`tuple`，存放了`(Path("*"), label)`
- 测试集只有数据没有标签

In [ ]:
from pathlib import Path

DATA_ROOT = Path(
    r"E:\projects\PyCharmProject\rookie\machine_learning\CV\ResNet"
    r"\Tiny_Images\tiny-imagenet-200"
)

TRAIN_ROOT = DATA_ROOT / "train"
VAL_ROOT = DATA_ROOT / "val"
TEST_ROOT = DATA_ROOT / "test"

- 用 `winds.txt` 是训练集的映射
- `class_to_idx`是train文件夹下每个类别标签和图片集的映射

In [ ]:
# 1. 以 wnids.txt 的顺序建立唯一、固定的类别映射
wnids = []
text = (DATA_ROOT / "wnids.txt").read_text(encoding="utf-8")
for line in text.splitlines():
    stripped = line.strip()
    if stripped:              # 非空才加入
        wnids.append(stripped)

class_to_idx = {wnid: idx for idx, wnid in enumerate(wnids)}

In [ ]:
len(class_to_idx)

- 训练集清单

In [ ]:
# 2. 训练集清单：
# train/<wnid>/images/xxx.JPEG
# p.parent       = images
# p.parent.parent = wnid
# 现在train_sample里面的数据是(path, label)，共100,000个数据（200*500）
train_samples = []
for path in sorted(TRAIN_ROOT.glob("*/*/*.JPEG")):
    wnid = path.parent.parent.name
    label = class_to_idx[wnid]
    train_samples.append((path.relative_to(DATA_ROOT), label))
train_samples

- 验证集清单

In [ ]:
# 3. 验证集清单：先读取文件名到 WNID 的映射
val_annotation = {}
annotation_path = VAL_ROOT / "val_annotations.txt"

for line in annotation_path.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue

    fields = line.split("\t")
    image_name, wnid = fields[0], fields[1]
    val_annotation[image_name] = wnid

val_paths = sorted((VAL_ROOT / "images").glob("*.JPEG"))

In [ ]:
# 防止标注缺项或存在无标注文件
val_file_names = {path.name for path in val_paths}

val_samples = []
for path in val_paths:
    wnid = val_annotation[path.name]
    label = class_to_idx[wnid]
    val_samples.append((path.relative_to(DATA_ROOT), label))
val_samples

- 测试集清单

In [ ]:
# 4. 测试集没有标签
test_paths = [
    path.relative_to(DATA_ROOT)
    for path in sorted((TEST_ROOT / "images").glob("*.JPEG"))
]

## 处理数据

### Dataset 与 DataLoader

下面的代码只负责把样本清单转换成可训练的 Tensor，不包含模型和训练循环。

数据形状必须始终满足：

```text
单张 JPEG（最终统一为 RGB）    [64, 64, 3]   # PIL 的通道在后
ToTensor 之后                  [3, 64, 64]   # PyTorch 的通道在前
DataLoader 组 Batch 之后       [B, 3, 64, 64]
标签 Batch                     [B]
```

In [ ]:
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


# 当前数据的输入尺寸。ResNet 的 stem 设计也必须以这个尺寸为前提。
IMAGE_SIZE = 64

# 训练 Batch 可以稍小，给模型和显存留下空间。
TRAIN_BATCH_SIZE = 64

# 验证和测试没有反向传播，不需要保存梯度，可以使用更大的 Batch。
EVAL_BATCH_SIZE = 256

# Windows 下的 Jupyter 在多进程中序列化 notebook 中定义的 Dataset 时容易出问题。
# 因此先使用 0，保证接口稳定；在独立 .py 训练脚本中通常可以使用 4 或 8。
NUM_WORKERS = 0


# 这组统计量是对当前 train 清单中的 100,000 张图执行以下步骤后得到的：
# convert("RGB") -> [0, 1] 浮点值 -> 按通道流式累计 sum 与 sum_square。
# 验证集和测试集没有参与统计，避免数据泄漏；图像增强也没有参与统计。
#
# 每个通道使用：
#   mean = E[x]
#   std  = sqrt(E[x^2] - E[x]^2)
TRAIN_MEAN = (0.48023694, 0.44806704, 0.39750364)
TRAIN_STD = (0.27643643, 0.26886328, 0.28158993)

- Transformer
> 1. 随机裁剪（先填充，再裁剪），这样可以改变图片位置
> 2. 随机翻转（只有训练集才做）
> 3. 将image 转成 Tensor
> 4. 标准化

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(
        size=IMAGE_SIZE,
        padding=4,
        padding_mode="reflect",
    ), # 随机裁剪——先填充，再随机才回去
    transforms.RandomHorizontalFlip(p=0.5), # 0.5 概率左右翻转
    transforms.ToTensor(), # 维度重排——HWC → CHW
    transforms.Normalize(mean=TRAIN_MEAN, std=TRAIN_STD), # 标准化
])


# 验证集和测试集必须可复现，不能使用随机裁剪或随机翻转。
# 形状：[64, 64, 3] -> [3, 64, 64]，数值随后进行相同的逐通道标准化。
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=TRAIN_MEAN, std=TRAIN_STD),
])

print(train_transform)
print(eval_transform)

- 自定义Dataset

In [ ]:
class ImageClassificationDataset(Dataset):
    """把 (相对路径, 整数标签) 清单映射成 (Tensor, 标量标签)。"""

    def __init__(self, samples, data_root, transform):
        # samples 形如：
        # [(Path("train/wnid/images/x.JPEG"), 0), ...]
        # 这里只保存路径和标签，不提前把图片像素放入内存。
        self.samples = samples
        self.data_root = data_root
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        relative_path, label = self.samples[index]
        image_path = self.data_root / relative_path

        if not image_path.is_file():
            raise FileNotFoundError(f"找不到图片：{image_path}")

        with Image.open(image_path) as image:
            # 数据集里同时存在 RGB 和 L（灰度）模式。
            # 必须先转成 RGB，保证每个样本最终都是 [3, 64, 64]。
            image = image.convert("RGB")

            # transform 输出：[3, 64, 64]，dtype=torch.float32。
            image = self.transform(image)

        # CrossEntropyLoss 默认要求 target 为 int64，形状由 DataLoader 变成 [B]。
        target = torch.tensor(label, dtype=torch.long)
        return image, target


In [ ]:
class ImageTestDataset(Dataset):
    """测试集没有标签，因此返回 (图像 Tensor, 相对路径)。"""

    def __init__(self, paths, data_root, transform):
        self.paths = paths
        self.data_root = data_root
        self.transform = transform

        if len(self.paths) == 0:
            raise ValueError("paths 不能为空。")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        relative_path = self.paths[index]
        image_path = self.data_root / relative_path

        if not image_path.is_file():
            raise FileNotFoundError(f"找不到图片：{image_path}")

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        # 返回字符串而不是 Path，默认 collate_fn 可以直接把路径组成 Python list。
        return image, str(relative_path)

- 建立DataLoader

In [ ]:
def build_dataloaders(
    train_samples,
    val_samples,
    test_paths,
    *,
    data_root=DATA_ROOT,
    train_transform=train_transform,
    eval_transform=eval_transform,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    num_workers=NUM_WORKERS,
):
    """根据三份清单一次性创建 train/val/test DataLoader。"""

    train_dataset = ImageClassificationDataset(
        samples=train_samples,
        data_root=data_root,
        transform=train_transform,
    )
    val_dataset = ImageClassificationDataset(
        samples=val_samples,
        data_root=data_root,
        transform=eval_transform,
    )
    test_dataset = ImageTestDataset(
        paths=test_paths,
        data_root=data_root,
        transform=eval_transform,
    )

    # 只有训练集需要 shuffle。验证集和测试集保持固定顺序，
    # 这样预测结果才能和路径、标签稳定对应。
    common_loader_args = {
        "num_workers": num_workers, # 用几个子进程加载数据
        "pin_memory": torch.cuda.is_available(), # 用锁页内存加速主机→显存拷贝
        "persistent_workers": num_workers > 0, # 让 worker 进程跨 epoch 存活
    }

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        drop_last=False,
        **common_loader_args,
    )
    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        **common_loader_args,
    )
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        **common_loader_args,
    )

    return train_loader, val_loader, test_loader


In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    train_samples=train_samples,
    val_samples=val_samples,
    test_paths=test_paths,
)

print(f"train: {len(train_loader.dataset):,} samples, {len(train_loader)} batches")
print(f"val:   {len(val_loader.dataset):,} samples, {len(val_loader)} batches")
print(f"test:  {len(test_loader.dataset):,} samples, {len(test_loader)} batches")
type(train_loader)

In [ ]:
idx = 0
for data in enumerate(train_loader):
    print(f"train_loader类型是{type(train_loader).__name__}")
    print(f"里面的数据是{type(data).__name__}, 有{len(data)}个元素")
    label = data[0]
    print(f"- data[0]:{type(data[0]).__name__},即label:{label}")

    data_list = data[1]
    print(f"- data[1]:{type(data[1]).__name__}, 存放了{len(data[1])}个tensor，分别是")

    data_shape = data_list[0]
    print(f"- - data_list[0].shape:{data_shape.shape}——一批的全部数据")
    print(f"{data_shape.shape[0]}张图片，{data_shape.shape[1]}个通道，width * height = {data_shape.shape[2]}*{data_shape.shape[3]}")

    data_shape = data_list[1]
    print(f"- - data_list[1].shape:{data_shape.shape}——一批的标签")

    print(sorted(set(data_shape)))

    break

# 搭建模型
- 输入：[Batch_size, 3, 64, 64]
- 输出：[Batch_size, 200]
```
ResNet
├── Stem Conv3x3, 3 -> 64, stride=1
├── Stage 1: 64 通道,  2 个 BasicBlock, 64x64
├── Stage 2: 128 通道, 2 个 BasicBlock, 32x32
├── Stage 3: 256 通道, 2 个 BasicBlock, 16x16
├── Stage 4: 512 通道, 2 个 BasicBlock, 8x8
├── AdaptiveAvgPool2d(1)
├── Flatten
└── Linear(512, 200)
```

## BasicBlock
- 关于`bn2`的权重偏置初始化
- `batchNormal`
> 1. 归一化（帮助训练）
> 2. 仿射变化（赋予网络自由度）——$y = \gamma \cdot \hat{x} + \beta$
- 起始 $\gamma, \beta$ 分别为1,0
- 分别设为0，让网络一开始就表现得像一个恒等映射

In [ ]:
from torch import nn
class BasicBlock(nn.Module):

    expansion = 1 # 最终输出通道数相对于基础宽度扩大了多少倍

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # stride=1：保持空间尺寸。
        # stride=2：空间尺寸减半，通常只出现在每个 Stage 的第一个 Block。
        if stride not in (1, 2):
            raise ValueError(f"stride 必须是 1 或 2，当前为 {stride}")

        # stride=1 输出：[B, out_channels, H, W]
        # stride=2 输出：[B, out_channels, H/2, W/2]
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # 这一层卷积不再改变空间
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 由于旁路需要保证和主通道一样的形状，所以这里可能还需要映射
        # 当stride = 2的时候 || 当输入通道数和输出通道数不一样的时候
        # 都需要映射

        need_projection = stride == 2 or in_channels != out_channels
        if need_projection:
            self.downsample = nn.Sequential(
                # kernel_size=1：只改变通道数，不引入额外空间信息
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, padding=0, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.downsample = nn.Identity()

        # 初始化bn2的weight和bias

        nn.init.zeros_(self.bn2.weight)
        nn.init.zeros_(self.bn2.bias)

    def forward(self, x):
        # out = conv1 -> bn1 -> relu -> conv2 -> bn2
        identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 主分支和旁路必须具有完全相同的 [B, C, H, W] 才能逐元素相加。
        out += identity

        out = self.relu(out)
        return out


## stage——多个 BasicBlock 的组合

| Stage | 输入形状 | 输出形状 | 首块 stride |
|---|---|---|---:|
| Stage 1 | `[B,64,64,64]` | `[B,64,64,64]` | 1 |
| Stage 2 | `[B,64,64,64]` | `[B,128,32,32]` | 2 |
| Stage 3 | `[B,128,32,32]` | `[B,256,16,16]` | 2 |
| Stage 4 | `[B,256,16,16]` | `[B,512,8,8]` | 2 |

In [ ]:
import torch
from torch import nn


def make_stage(
    in_channels,
    out_channels,
    num_blocks,
    first_stride,
):
    """用多个 BasicBlock 构建一个 ResNet Stage。"""

    if num_blocks < 1:
        raise ValueError("num_blocks 必须至少为 1")

    if first_stride not in (1, 2):
        raise ValueError("first_stride 必须是 1 或 2")

    blocks = []

    # Stage 的第一个 Block 负责可能的通道变化和空间降采样。
    #
    # first_stride=1：
    # [B, in_channels, H, W] -> [B, out_channels, H, W]
    #
    # first_stride=2：
    # [B, in_channels, H, W] -> [B, out_channels, H/2, W/2]
    blocks.append(
        BasicBlock(
            in_channels=in_channels,
            out_channels=out_channels,
            stride=first_stride,
        )
    )

    # 后续 Block 的输入和输出通道都是 out_channels，
    # 空间尺寸也不再变化。
    for _ in range(1, num_blocks):
        blocks.append(
            BasicBlock(
                in_channels=out_channels,
                out_channels=out_channels,
                stride=1,
            )
        )

    # nn.Sequential 会依次调用每个 BasicBlock，
    # 并把前一个 Block 的输出直接作为后一个 Block 的输入。
    return nn.Sequential(*blocks)

## 搭建网络

In [ ]:
class ResNetTiny(nn.Module):
    """面向 64x64 Tiny ImageNet 的 ResNet-18 网络"""
    def __init__(self, num_classes=200):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),
        )
        # 对stem里的卷积层进行He初始化
        for module in self.stem.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_out",
                    nonlinearity="relu",
                )

        self.stage_1 = make_stage(in_channels=64, out_channels=64, num_blocks=2, first_stride=1)
        self.stage_2 = make_stage(in_channels=64, out_channels=128, num_blocks=2, first_stride=2)
        self.stage_3 = make_stage(in_channels=128, out_channels=256, num_blocks=2, first_stride=2)
        self.stage_4 = make_stage(in_channels=256, out_channels=512, num_blocks=2, first_stride=2)

        self.pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.fc = nn.Linear(in_features=512, out_features=num_classes)
    def forward(self, x):
        # x——[BatchSize, 3, 64, 64]

        x = self.stem(x) # [BatchSize, 64, 64, 64]

        x = self.stage_1(x) # [BatchSize, 64, 64, 64]
        x = self.stage_2(x) # [BatchSize, 128, 32, 32]
        x = self.stage_3(x) # [BatchSize, 256, 16, 16]
        x = self.stage_4(x) # [BatchSize, 512, 8, 8]
        x = self.pool(x) # [BatchSize, 512, 1, 1]

        x = torch.flatten(x, start_dim=1) # [B, 512]

        logits = self.fc(x) # [B, 200]
        return logits

# 开始训练

## 基本准备
> 1. device
> 2. 数据和标签
> 3. 模型

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

model = ResNetTiny(num_classes=200).to(device)

In [ ]:
images, labels = next(iter(train_loader))
images = images.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True)

print(images.shape)   # [B, 3, 64, 64]
print(labels.shape)   # [B]

In [ ]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn


# ==================== 基础配置 ====================

SEED = 42
EPOCHS = 80

BATCH_SIZE = 128
BASE_LR = 0.05
MIN_LR = 1e-5
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1

WARMUP_EPOCHS = 3
MAX_GRAD_NORM = 5.0
EARLY_STOP_PATIENCE = 15

RUN_DIR = Path("runs/resnet_tiny_imagenet")
LAST_CHECKPOINT = RUN_DIR / "last.pth"
BEST_CHECKPOINT = RUN_DIR / "best.pth"
METRICS_FILE = RUN_DIR / "metrics.jsonl"

RUN_DIR.mkdir(parents=True, exist_ok=True)


# ==================== 随机种子 ====================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)


# ==================== 设备 ====================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = device.type == "cuda"
amp_device_type = "cuda" if device.type == "cuda" else "cpu"

print("device:", device)
print("amp_enabled:", amp_enabled)

if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    print("cuda version:", torch.version.cuda)

model = model.to(device)


# ==================== 损失、优化器和调度器 ====================

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=BASE_LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=True,
)


def lr_lambda(epoch):
    """
    epoch 是 LambdaLR 内部的零基 epoch 索引。

    前 WARMUP_EPOCHS 个 epoch 线性预热，
    之后按余弦曲线从 BASE_LR 下降到 MIN_LR。
    """
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / WARMUP_EPOCHS

    progress = (epoch - WARMUP_EPOCHS) / max(
        1, EPOCHS - WARMUP_EPOCHS
    )
    min_ratio = MIN_LR / BASE_LR

    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_ratio + (1.0 - min_ratio) * cosine


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_lambda,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled,
)

In [ ]:
def accuracy_counts(logits, targets):
    """返回 Top-1 和 Top-5 的正确样本数。"""
    top1_predictions = logits.argmax(dim=1)

    k = min(5, logits.shape[1])
    topk_predictions = logits.topk(k, dim=1).indices

    correct_top1 = (
        top1_predictions == targets
    ).sum().item()

    correct_top5 = topk_predictions.eq(
        targets.unsqueeze(1)
    ).any(dim=1).sum().item()

    return correct_top1, correct_top5


def train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    scaler,
    device,
    amp_enabled,
    max_grad_norm,
):
    model.train()

    total_loss = 0.0
    total_correct_top1 = 0
    total_correct_top5 = 0
    total_samples = 0
    grad_norm_sum = 0.0
    num_steps = 0

    start_time = time.perf_counter()

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        batch_size = labels.size(0)

        optimizer.zero_grad(set_to_none=True)

        # [B,3,64,64] -> [B,200]
        with torch.amp.autocast(
            device_type="cuda" if amp_enabled else "cpu",
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        # 梯度裁剪必须在 scaler.step() 之前执行。
        scaler.unscale_(optimizer)
        grad_norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=max_grad_norm,
        )

        scaler.step(optimizer)
        scaler.update()

        correct_top1, correct_top5 = accuracy_counts(
            logits.detach(),
            labels,
        )

        total_loss += loss.item() * batch_size
        total_correct_top1 += correct_top1
        total_correct_top5 += correct_top5
        total_samples += batch_size
        grad_norm_sum += grad_norm.item()
        num_steps += 1

    elapsed = time.perf_counter() - start_time

    metrics = {
        "loss": total_loss / total_samples,
        "top1": total_correct_top1 / total_samples,
        "top5": total_correct_top5 / total_samples,
        "grad_norm": grad_norm_sum / max(1, num_steps),
        "seconds": elapsed,
    }

    return metrics


@torch.inference_mode()
def evaluate(
    model,
    val_loader,
    criterion,
    device,
    amp_enabled,
):
    model.eval()

    total_loss = 0.0
    total_correct_top1 = 0
    total_correct_top5 = 0
    total_samples = 0

    start_time = time.perf_counter()

    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        batch_size = labels.size(0)

        with torch.amp.autocast(
            device_type="cuda" if amp_enabled else "cpu",
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        correct_top1, correct_top5 = accuracy_counts(
            logits,
            labels,
        )

        total_loss += loss.item() * batch_size
        total_correct_top1 += correct_top1
        total_correct_top5 += correct_top5
        total_samples += batch_size

    elapsed = time.perf_counter() - start_time

    metrics = {
        "loss": total_loss / total_samples,
        "top1": total_correct_top1 / total_samples,
        "top5": total_correct_top5 / total_samples,
        "seconds": elapsed,
    }

    return metrics

In [ ]:
def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    best_val_top1,
    no_improve_epochs,
):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_top1": best_val_top1,
        "no_improve_epochs": no_improve_epochs,
        "config": {
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "base_lr": BASE_LR,
            "min_lr": MIN_LR,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "seed": SEED,
        },
    }

    # 先写临时文件再替换，避免中断时把旧 checkpoint 写坏。
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary_path)
    temporary_path.replace(path)


def load_checkpoint(path):
    checkpoint = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_top1 = checkpoint["best_val_top1"]
    no_improve_epochs = checkpoint["no_improve_epochs"]

    return start_epoch, best_val_top1, no_improve_epochs

In [ ]:
start_epoch = 0
best_val_top1 = 0.0
no_improve_epochs = 0

# 如果 last.pth 已存在，则自动从上次中断处继续。
if LAST_CHECKPOINT.exists():
    print(f"Resuming from {LAST_CHECKPOINT}")
    start_epoch, best_val_top1, no_improve_epochs = (
        load_checkpoint(LAST_CHECKPOINT)
    )

print("start epoch:", start_epoch)
print("best val top1:", best_val_top1)


for epoch in range(start_epoch, EPOCHS):
    current_lr = optimizer.param_groups[0]["lr"]

    train_metrics = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
        amp_enabled=amp_enabled,
        max_grad_norm=MAX_GRAD_NORM,
    )

    val_metrics = evaluate(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=device,
        amp_enabled=amp_enabled,
    )

    improved = val_metrics["top1"] > best_val_top1

    if improved:
        best_val_top1 = val_metrics["top1"]
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1

    # 每个 epoch 更新一次 CosineAnnealing。
    scheduler.step()

    save_checkpoint(
        path=LAST_CHECKPOINT,
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        best_val_top1=best_val_top1,
        no_improve_epochs=no_improve_epochs,
    )

    if improved:
        save_checkpoint(
            path=BEST_CHECKPOINT,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            best_val_top1=best_val_top1,
            no_improve_epochs=no_improve_epochs,
        )

    record = {
        "epoch": epoch + 1,
        "learning_rate": current_lr,
        "train_loss": train_metrics["loss"],
        "train_top1": train_metrics["top1"],
        "train_top5": train_metrics["top5"],
        "train_grad_norm": train_metrics["grad_norm"],
        "train_seconds": train_metrics["seconds"],
        "val_loss": val_metrics["loss"],
        "val_top1": val_metrics["top1"],
        "val_top5": val_metrics["top5"],
        "val_seconds": val_metrics["seconds"],
    }

    with METRICS_FILE.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(
        f"Epoch {epoch + 1:03d}/{EPOCHS} | "
        f"lr={current_lr:.6f} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"train_top1={train_metrics['top1']:.4%} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_top1={val_metrics['top1']:.4%} | "
        f"val_top5={val_metrics['top5']:.4%} | "
        f"best={best_val_top1:.4%}"
    )

    if no_improve_epochs >= EARLY_STOP_PATIENCE:
        print(
            f"Early stopping: validation Top-1 "
            f"has not improved for {EARLY_STOP_PATIENCE} epochs."
        )
        break

Epoch 003/80 | lr=0.050000 | train_loss=3.7599 | train_top1=22.9290% | val_loss=3.6544 | val_top1=25.7800% | val_top5=52.6300% | best=25.7800%


In [ ]:
best_checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(best_checkpoint["model_state_dict"])

final_val_metrics = evaluate(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device,
    amp_enabled=amp_enabled,
)

print(
    f"Best epoch: {best_checkpoint['epoch'] + 1}\n"
    f"Val loss: {final_val_metrics['loss']:.4f}\n"
    f"Val Top-1: {final_val_metrics['top1']:.4%}\n"
    f"Val Top-5: {final_val_metrics['top5']:.4%}"
)